# 04. Baselines

This notebook quantifies how much performance is achievable without a learned neural perturbation model. Strong baselines are essential because they show whether a complex model truly captures structure beyond average residual behavior. The current pipeline already includes global-mean and cell-specific mean residual baselines; this notebook preserves those baselines and extends the benchmark with control-copy and dose-naive variants.


In [2]:
!pip -q install anndata scanpy scikit-learn scipy pandas numpy torch pyarrow

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import urllib.request
from pathlib import Path

HELPER_DIR = Path("/content/drive/MyDrive/ChemDFM")
if str(HELPER_DIR) not in sys.path:
    sys.path.append(str(HELPER_DIR))

from chemdfm_notebook_helpers import *

DATA_PATH = Path("/content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad")
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    print("Downloading SciPlex dataset...")
    URL = "https://f003.backblazeb2.com/file/chemCPA-datasets/sciplex_complete_middle_subset.h5ad"
    urllib.request.urlretrieve(URL, DATA_PATH)
    print("Download complete.")

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_PATH =", DATA_PATH)
print("Using device:", DEVICE)

Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/ChemDFM_repo
DATA_PATH = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad
Using device: cpu


In [3]:
import sys
from pathlib import Path

_support_candidates = [
    Path.cwd() / "notebooks_support",
    Path.cwd().parent / "notebooks_support",
    Path("/content/drive/MyDrive/ChemDFM/notebooks_support"),
]
for _cand in _support_candidates:
    if _cand.exists():
        sys.path.append(str(_cand))
        break

from chemdfm_notebook_helpers import *
print("Using device:", DEVICE)


Using device: cpu


In [6]:
from pathlib import Path
import json, os, random, pickle, warnings
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("/content/drive/MyDrive/ChemDFM")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
EXTERNAL_DIR = DATA_DIR / "external"
RUNS_DIR = PROJECT_ROOT / "runs"
RESULTS_DIR = PROJECT_ROOT / "results"
REPORTS_DIR = PROJECT_ROOT / "reports"

for p in [DATA_DIR, RAW_DIR, INTERIM_DIR, PROCESSED_DIR, EXTERNAL_DIR, RUNS_DIR, RESULTS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DEFAULT_H5AD = RAW_DIR / "sciplex_complete_middle_subset.h5ad"

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def map_split(x: str) -> str:
    x = str(x).lower()
    if "train" in x:
        return "train"
    if "test" in x or "val" in x:
        return "test"
    if "ood" in x:
        return "ood"
    return "drop"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DEFAULT_H5AD =", DEFAULT_H5AD)


import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, average_precision_score
from scipy.stats import pearsonr, spearmanr

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Using device:", DEVICE)


PROJECT_ROOT = /content/drive/MyDrive/ChemDFM
DEFAULT_H5AD = /content/drive/MyDrive/ChemDFM/data/raw/sciplex_complete_middle_subset.h5ad
Using device: cpu


In [7]:
from pathlib import Path
SPLIT_COL = "split_ho_pathway"
PCA_DIM = 25

adata, X = load_adata(DEFAULT_H5AD, split_col=SPLIT_COL)
obs = pd.read_parquet(PROCESSED_DIR / "datasets" / "adata_obs_processed.parquet")
adata.obs = obs.copy()

X_pca = np.load(PROCESSED_DIR / "pca" / f"X_pca_{PCA_DIM}d.npy")
X0_pca = np.load(PROCESSED_DIR / "datasets" / f"X0_pca_{PCA_DIM}d.npy")
DELTA_pca = np.load(PROCESSED_DIR / "datasets" / f"DELTA_pca_{PCA_DIM}d.npy")


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [8]:
train_mask = (adata.obs["_split3"] == "train").values
train_obs = adata.obs.iloc[np.where(train_mask)[0]].copy()
train_delta = DELTA_pca[train_mask]
train_pert_mask = train_mask & (adata.obs["condition"].astype(str).values != "control")

global_mean_delta = train_delta[train_obs["condition"].astype(str).values != "control"].mean(axis=0).astype(np.float32)

cell_mean_delta = {}
for cell in sorted(train_obs["cell_type"].astype(str).unique()):
    m = (
        (adata.obs["_split3"].values == "train")
        & (adata.obs["cell_type"].astype(str).values == cell)
        & (adata.obs["condition"].astype(str).values != "control")
    )
    cell_mean_delta[cell] = DELTA_pca[m].mean(axis=0).astype(np.float32) if m.sum() > 0 else global_mean_delta.copy()

drug_mean_delta = {}
for cond in sorted(train_obs["condition"].astype(str).unique()):
    if cond == "control":
        continue
    m = (
        (adata.obs["_split3"].values == "train")
        & (adata.obs["condition"].astype(str).values == cond)
        & (adata.obs["condition"].astype(str).values != "control")
    )
    if m.sum() > 0:
        drug_mean_delta[cond] = DELTA_pca[m].mean(axis=0).astype(np.float32)


## Baseline definitions

- **control_copy** predicts the control anchor directly.
- **global_mean** adds the global mean perturbation residual to the control anchor.
- **cell_mean** adds a cell-specific mean perturbation residual.
- **drug_lookup** uses the training-set mean residual for the same condition when available; otherwise, it falls back to the global mean.


In [9]:
def eval_baseline(split_name, mode):
    mask = (adata.obs["_split3"].values == split_name) & (adata.obs["condition"].astype(str).values != "control")
    idxs = np.where(mask)[0]
    preds, trues, x0s, conds, cells = [], [], [], [], []
    for idx in idxs:
        row = adata.obs.iloc[idx]
        cond = str(row["condition"])
        cell = str(row["cell_type"])
        x0 = X0_pca[idx]
        true = X_pca[idx]
        if mode == "control_copy":
            pred = x0
        elif mode == "global_mean":
            pred = x0 + global_mean_delta
        elif mode == "cell_mean":
            pred = x0 + cell_mean_delta[cell]
        elif mode == "drug_lookup":
            pred = x0 + drug_mean_delta.get(cond, global_mean_delta)
        else:
            raise ValueError(mode)
        preds.append(pred); trues.append(true); x0s.append(x0); conds.append(cond); cells.append(cell)
    pred = np.stack(preds); true = np.stack(trues); x0 = np.stack(x0s)
    overall = {"split": split_name, "mode": mode, **compute_metrics(pred, true, x0)}
    per_cell_rows = []
    for cell in sorted(set(cells)):
        m = np.array(cells) == cell
        per_cell_rows.append({"split": split_name, "mode": mode, "cell_type": cell, **compute_metrics(pred[m], true[m], x0[m])})
    return overall, pd.DataFrame(per_cell_rows)

overall_rows, per_cell_frames = [], []
for split_name in ["test", "ood"]:
    for mode in ["control_copy", "global_mean", "cell_mean", "drug_lookup"]:
        overall, per_cell = eval_baseline(split_name, mode)
        overall_rows.append(overall)
        per_cell_frames.append(per_cell)

baseline_overall = pd.DataFrame(overall_rows)
baseline_per_cell = pd.concat(per_cell_frames, ignore_index=True)

(RESULTS_DIR / "canonical").mkdir(parents=True, exist_ok=True)
baseline_overall.to_csv(RESULTS_DIR / "canonical" / "baseline_metrics_overall.csv", index=False)
baseline_per_cell.to_csv(RESULTS_DIR / "canonical" / "baseline_metrics_per_cell.csv", index=False)

baseline_overall.sort_values(["split", "r2_top50"], ascending=[True, False])


,split,mode,r2_full,pearson_rowmean,mse,collapse_ratio,mean_shift_error,r2_top20,r2_top50
7,ood,drug_lookup,0.571480,0.747878,0.399335,0.651541,0.466813,0.442601,0.520211
6,ood,cell_mean,0.563591,0.739673,0.406687,0.612709,0.469147,0.436299,0.514799
5,ood,global_mean,0.559701,0.738252,0.410312,0.644630,0.470395,0.430381,0.509394
4,ood,control_copy,0.554779,0.734811,0.414899,0.645817,0.471575,0.424265,0.503718
3,test,drug_lookup,0.617675,0.774740,0.358401,0.648411,0.450282,0.475879,0.557363
2,test,cell_mean,0.603489,0.763246,0.371700,0.602134,0.455494,0.464252,0.547430
1,test,global_mean,0.602405,0.762905,0.372716,0.633616,0.455401,0.461525,0.544917
0,test,control_copy,0.600740,0.761599,0.374277,0.635082,0.455507,0.458998,0.542528


These baselines define the minimum standard that any learned model must beat. The following notebook trains the simpler residual model so that subsequent improvements can be interpreted as component-wise gains rather than unexplained code drift.
